# Tome 6, Chapter 10: Differentiation and Radon–Nikodym (Python companion)

Exercises first, solutions at the end. Densities and derivatives on a computer: importance sampling of a Gaussian tail with a Radon–Nikodym weight, rare events for coin tosses by exponential tilting, Salem's strictly increasing function with derivative zero, and maximal functions and Lebesgue points. Requirements: numpy, scipy and matplotlib.

*Before each solution, predict what the output will look like.*

## Exercises

**P10.1 (★★) CS application: importance sampling.** Estimate $p=\mathbb P(Z>5)$ for a standard Gaussian $Z$ (about $2.9\cdot10^{-7}$) with $10^5$ samples: (a) by plain Monte Carlo; (b) by sampling $Y\sim\mathcal N(\theta,1)$ and averaging $\mathbf 1_{Y>5}\,\frac{d\mathcal N(0,1)}{d\mathcal N(\theta,1)}(Y)$, for several $\theta$. Compute the variance of the estimator as a function of $\theta$ and find the best shift. Why is the estimator unbiased (Theorem 10.4)?

**P10.2 (★★) Rare events for coin tosses by exponential tilting.** Estimate $\mathbb P(S_n\ge0.7n)$ for $n=200$ fair coins by sampling biased coins of parameter $q$ and reweighting with the likelihood ratio. Compare with the exact binomial tail and with the Chernoff bound $e^{-nI(0.7)}$, $I(a)=a\ln(2a)+(1-a)\ln(2(1-a))$.

**P10.3 (★) A strictly increasing function with derivative zero.** Plot Salem's function $F_p$ of Exercise 10.4 for $p=0.3$ (sample $X$ and plot the empirical distribution function, or compute $F_p$ on dyadic points exactly). Estimate $\frac{F_p(x+h)-F_p(x)}h$ at random $x$ for $h=2^{-10},2^{-20},2^{-30}$, and explain the result with the frequencies of binary digits.

**P10.4 (★) Maximal functions in practice.** For $f(x)=|x|^{-1/2}\mathbf 1_{|x|<1}$, compute the maximal function on a grid, check the maximal inequality $t\,\lambda(Mf>t)\le3\|f\|_1$ (Theorem 10.8), and estimate the best constant $\sup_t t\lambda(Mf>t)/\|f\|_1$ for this $f$. Then check Lebesgue's differentiation theorem at a few points, including $0$.

## Solutions

### P10.1

In [ ]:
import numpy as np
from scipy.stats import norm
rng = np.random.default_rng(0)
n = 100_000
p = norm.sf(5)
print(f"exact p = {p:.4e}")
Z = rng.standard_normal(n)
print("plain Monte Carlo:", (Z > 5).mean())
for theta in [3, 4, 5, 5.2, 6, 7]:
    Y = rng.standard_normal(n) + theta
    w = np.exp(-theta * Y + theta ** 2 / 2)                # dN(0,1)/dN(theta,1) at Y
    est = np.mean((Y > 5) * w)
    sd = np.std((Y > 5) * w) / np.sqrt(n)
    print(f"theta={theta:3.1f}: estimate {est:.4e}   relative standard error {sd/p:.3f}")
# exact second moment: E_theta[1_{Y>5} w^2] = e^{theta^2} P(Z > 5 + theta)
th = np.linspace(3, 8, 501)
rel = np.sqrt(np.exp(th ** 2) * norm.sf(5 + th) - p ** 2) / p
print("best theta (exact variance):", th[np.argmin(rel)], " relative error per sample:", rel.min())

Plain Monte Carlo sees on average $0.03$ successes in $10^5$ trials, so its estimate is almost always $0$: rare events are invisible. Sampling from $\mathbb Q=\mathcal N(\theta,1)$ makes the event common, and the weight $L=\frac{d\mathbb P}{d\mathbb Q}(y)=e^{-\theta y+\theta^2/2}$ corrects the bias exactly: $\mathbb E_{\mathbb Q}[\mathbf 1_{Y>5}L(Y)]=\mathbb P(Z>5)$ because $\mathbb P\ll\mathbb Q$ and $L$ is the Radon–Nikodym derivative (Theorem 10.4). The second moment is $e^{\theta^2}\mathbb P(Z>5+\theta)$, minimised near $\theta\approx5.1$, slightly beyond the threshold: the best sampling law puts the mass where the event is likely *and* the weights are not too wild. With it, $10^5$ samples give a relative error below $1\%$, a variance reduction by a factor of more than $10^5$ (per sample, the relative variance drops from $\frac{1-p}p\approx3.5\cdot10^6$ to about $5.6$). A bad $\theta$ (too large) blows up the weights, and the estimator's error grows again.

### P10.2

In [ ]:
import numpy as np
from scipy.stats import binom
rng = np.random.default_rng(1)
n, a, N = 200, 0.7, 100_000
k0 = int(np.ceil(a * n))
exact = binom.sf(k0 - 1, n, 0.5)
I = a * np.log(2 * a) + (1 - a) * np.log(2 * (1 - a))
print(f"exact {exact:.4e}   Chernoff bound {np.exp(-n * I):.4e}")
print("plain Monte Carlo:", (rng.binomial(n, 0.5, N) >= k0).mean())
for q in [0.6, 0.7, 0.8]:
    S = rng.binomial(n, q, N)
    w = (0.5 / q) ** S * (0.5 / (1 - q)) ** (n - S)         # likelihood ratio of the whole sequence depends only on S
    est = np.mean((S >= k0) * w)
    print(f"q={q}: estimate {est:.4e}   relative error {np.std((S >= k0) * w) / np.sqrt(N) / exact:.3f}")

The probability is about $7.5\cdot10^{-9}$, out of reach of plain Monte Carlo. Under biased coins of parameter $q$, the likelihood ratio of a sequence with $S$ ones is $\bigl(\frac{1/2}q\bigr)^S\bigl(\frac{1/2}{1-q}\bigr)^{n-S}$, which depends on the sequence only through $S$. Tilting to $q=a=0.7$, the value that makes the rare event typical, gives a relative error of about $1\%$; this is *exponential tilting*, and the same change of measure proves Cramér's theorem on large deviations (Tome 10). The Chernoff bound $e^{-nI(a)}$ has the right exponential rate but overestimates, here by a factor of about $10$: the missing factor is of order $\frac1{\sqrt n}$.

### P10.3

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
p = 0.3
def F(x, depth=60):
    # F_p(x) = P(X <= x), X = sum B_k 2^-k with P(B_k = 1) = p: read the binary digits of x
    acc, mass = 0.0, 1.0
    for _ in range(depth):
        x *= 2; d = int(x); x -= d
        if d == 1:
            acc += mass * (1 - p)          # all X whose next bit is 0 lie below x
            mass *= p
        else:
            mass *= 1 - p
    return acc
xs = np.linspace(0, 1, 2001)
plt.figure(figsize=(5, 4)); plt.plot(xs, [F(x) for x in xs]); plt.plot([0, 1], [0, 1], "k:", lw=0.7); plt.title("Salem's function, p = 0.3"); plt.show()
rng = np.random.default_rng(2)
# exact: on the dyadic interval of rank n containing x, F_p increases by p^k (1-p)^(n-k), k = number of ones among the first n digits
for trial in range(3):
    bits = rng.integers(0, 2, 10_000)                       # the binary digits of a uniformly random x
    out = []
    for n in [10, 100, 1000, 10_000]:
        k = bits[:n].sum()
        log10_ratio = (k * np.log10(2 * p) + (n - k) * np.log10(2 * (1 - p)))
        out.append(f"n={n}: 10^{log10_ratio:.1f}")
    print("increase / length over the rank-n interval:", "   ".join(out))
print("prediction (4p(1-p))^(n/2) for n = 10000: 10^%.1f" % (5000 * np.log10(4 * p * (1 - p))))

The function rises from $0$ to $1$, continuously and strictly. Floating-point difference quotients are too coarse to see its derivative vanish ($h\ge2^{-52}$), so the code works exactly with digits: over the dyadic interval of rank $n$ containing $x$, $F_p$ increases by $p^k(1-p)^{n-k}$, where $k$ is the number of ones among the first $n$ digits of $x$, while the interval has length $2^{-n}$. For a typical $x$, $k\approx\frac n2$ (Borel), so the ratio is about $(4p(1-p))^{n/2}$, up to a factor $e^{O(\sqrt n)}$ from the fluctuations of $k$, and $4p(1-p)=0.84<1$: at rank $10\,000$ the ratio is about $10^{-370}$, against the prediction $10^{-379}$. All the increase of $F_p$ happens on the null set of numbers whose digits have frequency $p$.

### P10.4

In [ ]:
import numpy as np
L, n = 8.0, 16001
x = np.linspace(-L, L, n); dx = x[1] - x[0]
with np.errstate(divide="ignore"):
    f = np.where(np.abs(x) < 1, np.abs(x) ** -0.5, 0.0)
f[n // 2] = 0.0                                            # the singular point carries no mass
norm1 = 4.0                                                # integral of |x|^-1/2 over (-1, 1)
# exact running integral of f: G(x) = int_{-1}^x f, using int |s|^-1/2 ds = 2 sgn(s) sqrt|s|
G = np.where(x <= -1, 0.0, np.where(x >= 1, 4.0, 2 + 2 * np.sign(x) * np.sqrt(np.abs(x))))
def Gint(a, b): return np.interp(b, x, G) - np.interp(a, x, G)
radii = np.geomspace(1e-4, 2 * L, 400)
Mf = np.zeros(n)
for r in radii:
    Mf = np.maximum(Mf, Gint(x - r, x + r) / (2 * r))
for t in [0.5, 1, 2, 5, 20]:
    meas = np.sum(Mf > t) * dx
    print(f"t={t:4}: t * lambda(Mf > t) = {t * meas:.3f}   3 ||f||_1 = {3 * norm1}")
print("ratio sup_t t lambda(Mf>t) / ||f||_1 ~", max(t * np.sum(Mf > t) * dx for t in np.linspace(0.3, 30, 300)) / norm1)
for x0 in [0.5, -0.3, 0.0]:
    avgs = [Gint(x0 - r, x0 + r) / (2 * r) for r in [1e-1, 1e-2, 1e-3, 1e-4]]
    print(f"averages around x={x0}: {np.round(avgs, 4)}   f(x) = {abs(x0) ** -0.5 if x0 else float('inf')}")

The maximal inequality holds with plenty of room: here $t\,\lambda(Mf>t)$ never exceeds about $3.4$, while $\|f\|_1=4$: the ratio is about $0.84$, far below the constant $3$ of the theorem (the best constant for the centred maximal function in dimension one is known to be $\frac{11+\sqrt{61}}{12}\approx1.57$, a theorem of Melas, 2003). The averages over shrinking intervals converge to $f(x)=|x|^{-1/2}$ at $x=0.5$ and $x=-0.3$, Lebesgue points, and blow up like $r^{-1/2}$ at $0$, where $f$ is infinite: a single point is a null set, and Lebesgue's theorem has nothing to say there.